# Prototype: dense retrieval, MMR and review evidence on Amazon ESCI

Goal: decide whether the scoring design (relevance x review evidence) is worth building out.

* **Data**: frozen subset of the Shopping Queries Dataset (ESCI, US, `small_version`, test split): 500-query pool, 20 eval queries, seed 42 (`scripts/prepare_prototype.py`).
* **Gains**: KDD Cup 2022 convention, E=1, S=0.1, C=0.01, I=0. Recall@10 counts only Exact products.
* **Two settings**: (A) *re-rank* the products judged for each query, the standard ESCI Task 1 setup; (B) *retrieve* from the pooled corpus of every product judged for the 500 queries (unjudged hits count as 0, so this underestimates absolute quality).
* **Models**: BM25, `BAAI/bge-small-en-v1.5` (general MTEB model) and `hyp1231/blair-roberta-base` (BLaIR, pretrained on Amazon Reviews 2023).

In [1]:
import json, os, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, AutoTokenizer
from pra.metrics import ESCI_GAIN, ndcg_at_k, recall_at_k, intra_list_diversity
from pra.retrieval import BM25Index, mmr

load_dotenv(Path('..') / '.env')
P = Path('../data/proto')
np.random.seed(42); torch.manual_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
queries = pd.read_parquet(P / 'queries.parquet')
qrels = pd.read_parquet(P / 'qrels.parquet')
corpus = pd.read_parquet(P / 'corpus.parquet').reset_index(drop=True)
print(DEVICE, len(queries), 'eval queries |', qrels.query_id.nunique(), 'pool queries |', len(corpus), 'products')
print(json.loads((P / 'overlap.json').read_text()))

cuda 20 eval queries | 500 pool queries | 10063 products
{'esci_us_test_products': 164900, 'appliances_items': 94327, 'overlap_products': 299, 'overlap_share': 0.0018, 'pool_products_with_reviews': 30, 'pool_products': 10063}


In [2]:
def _s(x):
    return x if isinstance(x, str) else ''

def product_text(r):
    return f"{_s(r.product_title)}. Brand: {_s(r.product_brand)}. {_s(r.product_bullet_point)[:600]}"

corpus['text'] = [product_text(r) for r in corpus.itertuples()]
pid_to_pos = {p: i for i, p in enumerate(corpus.product_id)}
gains = {q: {p: ESCI_GAIN[l] for p, l in zip(g.product_id, g.esci_label)} for q, g in qrels.groupby('query_id')}
exact = {q: {p for p, l in zip(g.product_id, g.esci_label) if l == 'E'} for q, g in qrels.groupby('query_id')}

In [3]:
# Embeddings
bge = SentenceTransformer('BAAI/bge-small-en-v1.5', device=DEVICE)
BGE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

blair_tok = AutoTokenizer.from_pretrained('hyp1231/blair-roberta-base')
blair = AutoModel.from_pretrained('hyp1231/blair-roberta-base').to(DEVICE).eval()

@torch.no_grad()
def blair_encode(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = blair_tok(texts[i:i+bs], padding=True, truncation=True, max_length=256, return_tensors='pt').to(DEVICE)
        v = blair(**enc).last_hidden_state[:, 0]  # first-token embedding, as in the model card
        out.append(torch.nn.functional.normalize(v, dim=-1).cpu().numpy())
    return np.concatenate(out)

t = time.time()
doc_vecs = {
    'bge-small': bge.encode(corpus.text.tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False),
    'blair-base': blair_encode(corpus.text.tolist()),
}
q_texts = queries['query'].tolist()
q_vecs = {
    'bge-small': bge.encode([BGE_QUERY_PREFIX + q for q in q_texts], normalize_embeddings=True),
    'blair-base': blair_encode(q_texts),
}
print(f'encoded in {time.time()-t:.0f}s')
bm25 = BM25Index(corpus.product_id.tolist(), corpus.text.tolist())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

encoded in 263s


In [4]:
def rank(method, qi, query, candidates=None, k=10):
    if method == 'bm25':
        return [p for p, _ in bm25.search(query, k=k, candidates=candidates)]
    D = doc_vecs[method]
    idx = np.asarray(candidates) if candidates is not None else np.arange(len(D))
    sims = D[idx] @ q_vecs[method][qi]
    return [corpus.product_id[i] for i in idx[np.argsort(-sims, kind='stable')[:k]]]

rows = []
for setting in ['A: re-rank judged', 'B: pooled retrieval']:
    for method in ['bm25', 'bge-small', 'blair-base']:
        nd, rc = [], []
        for qi, (qid, q) in enumerate(zip(queries.query_id, queries['query'])):
            cands = [pid_to_pos[p] for p in gains[qid] if p in pid_to_pos] if setting.startswith('A') else None
            r = rank(method, qi, q, cands)
            nd.append(ndcg_at_k(r, gains[qid])); rc.append(recall_at_k(r, exact[qid]))
        rows.append({'setting': setting, 'method': method, 'nDCG@10': np.mean(nd), 'recall@10 (E)': np.mean(rc)})
results = pd.DataFrame(rows).round(3)
results

,setting,method,nDCG@10,recall@10 (E)
0,A: re-rank judged,bm25,0.685,0.680
1,A: re-rank judged,bge-small,0.729,0.745
2,A: re-rank judged,blair-base,0.725,0.736
3,B: pooled retrieval,bm25,0.524,0.497
4,B: pooled retrieval,bge-small,0.561,0.548
5,B: pooled retrieval,blair-base,0.377,0.329


## MMR re-ranking
Take the top-50 by cosine from the best dense model in setting B, then re-rank with MMR. `lambda = 1` is plain cosine. Diversity = 1 - mean pairwise cosine of the top-10.

In [5]:
best = results[results.setting.str.startswith('B') & (results.method != 'bm25')].sort_values('nDCG@10').method.iloc[-1]
D = doc_vecs[best]
rows = []
for lam in [1.0, 0.85, 0.7, 0.5]:
    nd, div = [], []
    for qi, qid in enumerate(queries.query_id):
        qv = q_vecs[best][qi]
        top = np.argsort(-(D @ qv), kind='stable')[:50]
        order = top[mmr(qv, D[top], k=10, lam=lam)]
        nd.append(ndcg_at_k([corpus.product_id[i] for i in order], gains[qid]))
        div.append(intra_list_diversity(D[order]))
    rows.append({'model': best, 'lambda': lam, 'nDCG@10': np.mean(nd), 'diversity@10': np.mean(div)})
mmr_table = pd.DataFrame(rows).round(3)
mmr_table

,model,lambda,nDCG@10,diversity@10
0,bge-small,1.00,0.561,0.280
1,bge-small,0.85,0.508,0.314
2,bge-small,0.70,0.404,0.345
3,bge-small,0.50,0.265,0.382


## Review evidence (Claude Haiku 5.5, structured outputs)
Demo query: the ESCI test query with the most Exact products that also have reviews in Amazon Reviews 2023 (Appliances). Up to 20 reviews per product. Requires `ANTHROPIC_API_KEY` in `.env`; otherwise these cells are skipped.

In [6]:
from pra.reviews import extract_features, aggregate
demo = json.loads((P / 'demo.json').read_text())
demo_reviews = pd.read_parquet(P / 'demo_reviews.parquet')
print(demo['query'], '|', len(demo['products']), 'products |', len(demo_reviews), 'reviews')
HAVE_KEY = bool(os.getenv('ANTHROPIC_API_KEY'))
mined = {}
usage = {'input_tokens': 0, 'output_tokens': 0}
if HAVE_KEY:
    import anthropic
    client = anthropic.Anthropic()
    for pid, g in demo_reviews.groupby('product_id'):
        per = []
        for r in g.itertuples():
            rf, u = extract_features(client, f"{r.title}\n{r.text}")
            per.append(rf); usage = {k: usage[k] + u[k] for k in usage}
        mined[pid] = aggregate(per)
    # Haiku 5.5 list price: $0.10 / 1M input, $0.50 / 1M output (Batch API halves it)
    cost = usage['input_tokens'] / 1e6 * 0.10 + usage['output_tokens'] / 1e6 * 0.50
    print(usage, f'cost ${cost:.4f} -> ${cost / len(demo_reviews) * 1000:.3f} per 1,000 reviews')
else:
    print('No ANTHROPIC_API_KEY: skipping extraction')

areo press filters | 19 products | 304 reviews


ANTHROPIC_API_KEY is set and takes precedence over the SDK's profile / federation auto-discovery; unset ANTHROPIC_API_KEY to use the auto-discovered credential.


{'input_tokens': 214939, 'output_tokens': 48590} cost $0.0458 -> $0.151 per 1,000 reviews


In [7]:
if mined:
    top_pid = max(mined, key=lambda p: sum(sum(c.values()) for c in mined[p].values()))
    tbl = pd.DataFrame(mined[top_pid]).T
    tbl['mentions'] = tbl.sum(axis=1)
    display(tbl.sort_values('mentions', ascending=False).head(15))

,positive,negative,neutral,mentions
ease of cleaning,4,1,0,5
cleaning,2,1,1,4
flavor,3,0,0,3
price,2,1,0,3
sediment,1,2,0,3
durability,3,0,0,3
performance,2,0,1,3
fit,2,0,0,2
ease of use,2,0,0,2
sharpness,1,1,0,2


## Transparent score and ablation
`evidence = (pos + 1) / (pos + neg + 2)` over mentions whose feature is semantically close (cosine >= 0.6, bge-small) to a word in the query; if none match, all mentions are used. `score = relevance x evidence**alpha`; `alpha = 0` is the no-reviews ablation.

In [8]:
from pra.scoring import evidence_score, product_score
if mined:
    qv = bge.encode([BGE_QUERY_PREFIX + demo['query']], normalize_embeddings=True)[0]
    q_terms = [t for t in demo['query'].split() if len(t) > 2]
    term_vecs = bge.encode(q_terms, normalize_embeddings=True)
    dp = pd.read_parquet(P / 'demo_products.parquet')
    dp_vecs = dict(zip(dp.product_id, bge.encode([product_text(r) for r in dp.itertuples()], normalize_embeddings=True)))
    rows = []
    for pid in demo['products']:
        if pid not in dp_vecs or pid not in mined:
            continue
        rel = float(dp_vecs[pid] @ qv)
        feats = list(mined[pid])
        fv = bge.encode(feats, normalize_embeddings=True)
        close = [f for f, v in zip(feats, fv) if (term_vecs @ v).max() >= 0.6] or feats
        pos = sum(mined[pid][f]['positive'] for f in close); neg = sum(mined[pid][f]['negative'] for f in close)
        ev = evidence_score({'positive': pos, 'negative': neg})
        rows.append({'product_id': pid, 'label': demo['labels'][pid], 'relevance': rel, 'pos': pos, 'neg': neg,
                     'evidence': ev, 'score_a1': product_score(rel, ev, 1), 'score_a0': product_score(rel, ev, 0),
                     'matched_features': ', '.join(close[:5])})
    demo_tbl = pd.DataFrame(rows).round(3)
    g = {p: ESCI_GAIN[l] for p, l in demo['labels'].items()}
    for col in ['score_a0', 'score_a1']:
        print(col, 'nDCG@10 =', round(ndcg_at_k(demo_tbl.sort_values(col, ascending=False).product_id.tolist(), g), 3))
    r = demo_tbl.iloc[0]
    print(f"By hand, {r.product_id}: ({r.pos}+1)/({r.pos}+{r.neg}+2) = {(r.pos+1)/(r.pos+r.neg+2):.3f}; "
          f"{r.relevance:.3f} x {r.evidence:.3f} = {r.relevance*r.evidence:.3f}")
    display(demo_tbl)

score_a0 nDCG@10 = 0.733
score_a1 nDCG@10 = 0.82
By hand, B08DKDJ2ZY: (36+1)/(36+13+2) = 0.725; 0.632 x 0.725 = 0.458


,product_id,label,relevance,pos,neg,evidence,score_a1,score_a0,matched_features
0,B08DKDJ2ZY,E,0.632,36,13,0.725,0.459,0.632,"paper filters, quality, build quality, filter ..."
1,B08L37D7WV,E,0.635,34,10,0.761,0.483,0.635,"value, fit, price, filter, overall performance"
2,B081T4MKQ7,E,0.624,5,2,0.667,0.416,0.624,"filter replacements, filter quality, coffee gr..."
3,B08TBYPK4J,E,0.664,46,12,0.783,0.520,0.664,"price, overall performance, fit, cleaning, pap..."
4,B08NPKTW9D,C,0.664,24,3,0.862,0.572,0.664,"size, fit, price, bleach, filter movement"
5,B08QHY2GQ7,S,0.634,23,23,0.500,0.317,0.634,"filter quality, coffee taste, price, fit, perf..."
6,B07MP1DNLF,S,0.711,21,7,0.733,0.521,0.711,"performance, sample absorption, filter thickne..."
7,B07FSQ2LHW,C,0.604,21,9,0.688,0.415,0.604,"price, lid design, fit, storage organization, ..."
8,B01DCGO8YM,E,0.627,37,5,0.864,0.542,0.627,"filters, straws, price, filter supply, perform..."
9,B00E58P7ME,E,0.689,39,17,0.690,0.475,0.689,"body, paper filters, fines filtration, perform..."


## Reading the results
* Setting A is comparable in kind to the ESCI paper's Task 1 baselines; setting B is a harder, pooled-retrieval view.
* 20 queries is a smoke test, not a benchmark: differences under ~0.03 nDCG are noise. Story 1 freezes 500 queries.
* The review demo rests on one query, so the ablation shows the mechanics, not a general effect.